# Multi-source Domain Adaptation - Parte 1: Dati, Backbone, Feature

**Gruppo:** DataLost - **Track 9**

Questo notebook documenta la prima parte del progetto: l'analisi dei tre dataset,
la costruzione del backbone per l'estrazione delle feature, e la pipeline di
estrazione vera e propria.

E un notebook **espositivo**: mostra il codice realmente usato nel progetto e i
risultati ottenuti. L'estrazione completa delle feature gira su un cluster con GPU
(decine di minuti su migliaia di clip) e non è pensata per essere eseguita
interamente in modo interattivo; le celle pesanti sono quindi illustrate con i
risultati gia calcolati.

## Indice
1. Analisi dei dataset
2. Il backbone: ResNet-50 ImageNet "inflated" a 3D (I3D)
3. Pipeline di estrazione delle feature


### Setup

Il notebook usa i moduli del progetto (cartella `src/`). Questa cella imposta la
directory di lavoro alla **root del progetto**, cosi gli import `src.*` funzionano
anche se il notebook viene aperto dalla cartella `notebooks/`.


In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print("Directory di lavoro:", os.getcwd())


Directory di lavoro: c:\Users\Jonas\Desktop\DomainAdaptation-Track9-DataLost


## 1. Analisi dei dataset

Disponiamo di tre dataset di riconoscimento di azioni:

- **HMDB-51** (sorgente): clip come cartelle di frame JPG gia estratti.
- **UCF-101** (sorgente): 101 classi, clip video `.avi`, una cartella per classe.
- **Kinetics-400** (target, sottoinsieme `kinetics400_5per`): clip video `.mp4`,
  usate **senza etichette** in addestramento (le etichette servono solo per la
  valutazione finale).

### 1.1 Spazio di etichette condiviso

I tre dataset hanno tassonomie diverse. La domain adaptation closed-set richiede
uno spazio di etichette comune, quindi prendiamo l'**intersezione** delle tre
tassonomie. Ispirandoci al benchmark UCF-HMDB (Chen et al., 2019) otteniamo
**11 classi** condivise. La classe `fencing` del benchmark e stata esclusa perche
assente da Kinetics-400 (preferiamo 11 classi pulite a una mappatura forzata).


In [2]:
from src.data.class_mapping import SHARED_CLASSES, DATASET_NAME_MAPS, NUM_CLASSES

print(f"Numero di classi condivise: {NUM_CLASSES}")
print("Classi:", SHARED_CLASSES)


Numero di classi condivise: 11
Classi: ['climb', 'golf', 'kick_ball', 'pullup', 'punch', 'pushup', 'ride_bike', 'ride_horse', 'shoot_ball', 'shoot_bow', 'walk']


La mappatura tra il nome canonico e il nome grezzo in ciascun dataset e
definita esplicitamente. Alcuni accoppiamenti non banali (da motivare):
`climb` -> UCF `RopeClimbing` / Kinetics `climbing a rope`;
`kick_ball` -> `SoccerPenalty` / `kicking soccer ball`;
`walk` -> `WalkingWithDog` / `walking the dog` (accoppiamento piu debole).


In [3]:
# Tabella della mappatura tra i tre dataset
print(f"{'canonica':12s} | {'HMDB-51':14s} | {'UCF-101':16s} | Kinetics-400")
print("-" * 70)
for c in SHARED_CLASSES:
    h = DATASET_NAME_MAPS['hmdb51'][c]
    u = DATASET_NAME_MAPS['ucf101'][c]
    k = DATASET_NAME_MAPS['kinetics'][c]
    print(f"{c:12s} | {h:14s} | {u:16s} | {k}")


canonica     | HMDB-51        | UCF-101          | Kinetics-400
----------------------------------------------------------------------
climb        | climb          | RopeClimbing     | climbing a rope
golf         | golf           | GolfSwing        | golf driving
kick_ball    | kick_ball      | SoccerPenalty    | kicking soccer ball
pullup       | pullup         | PullUps          | pull ups
punch        | punch          | Punch            | punching person (boxing)
pushup       | pushup         | PushUps          | push up
ride_bike    | ride_bike      | Biking           | riding a bike
ride_horse   | ride_horse     | HorseRiding      | riding or walking with horse
shoot_ball   | shoot_ball     | Basketball       | shooting basketball
shoot_bow    | shoot_bow      | Archery          | archery
walk         | walk           | WalkingWithDog   | walking the dog


### 1.2 Statistiche dei dataset

Conteggio delle clip per dataset e per classe, dopo il filtraggio sulle 11 classi
condivise. Questi numeri sono ricavati dai file `index.csv` prodotti
dall'estrazione delle feature. Riportiamo i valori ottenuti.


In [4]:
# Statistiche reali ottenute dall'estrazione (dai file index.csv)
stats = {
    'HMDB-51':  {'totale': 1684, 'note': 'walk sovrarappresentata: 548 clip'},
    'UCF-101':  {'totale': 1457, 'note': 'ben bilanciato (100-164 per classe)'},
    'Kinetics': {'totale': 303,  'note': 'target ridotto e sbilanciato (12-45/classe)'},
}
print(f"{'Dataset':10s} | {'Clip':>5s} | Note")
print("-" * 60)
for d, s in stats.items():
    print(f"{d:10s} | {s['totale']:5d} | {s['note']}")


Dataset    |  Clip | Note
------------------------------------------------------------
HMDB-51    |  1684 | walk sovrarappresentata: 548 clip
UCF-101    |  1457 | ben bilanciato (100-164 per classe)
Kinetics   |   303 | target ridotto e sbilanciato (12-45/classe)


**Osservazioni rilevanti per il seguito del progetto:**

- In **HMDB-51** la classe `walk` è fortemente sovrarappresentata (548 clip contro
  ~110 delle altre). Questo introdurra un bias del classificatore verso `walk`,
  che analizzeremo e correggeremo nella seconda parte (bilanciamento di classe).
- Il **target Kinetics** è piccolo (303 clip) e sbilanciato: le metriche su alcune
  classi (es. `climb` con 12 clip) saranno poco robuste.
- **UCF-101** è il piu bilanciato dei tre.

Il codice seguente mostra come si calcolano questi conteggi a partire da un
`index.csv` (riproducibile sul cluster dopo l'estrazione).


In [5]:
# Come ricavare i conteggi per classe da un index.csv (esempio)
import csv
from collections import Counter
from src.data.class_mapping import IDX_TO_CLASS

def conta_per_classe(index_csv):
    with open(index_csv) as f:
        reader = csv.DictReader(f)
        c = Counter(int(r['label']) for r in reader)
    return {IDX_TO_CLASS[k]: v for k, v in sorted(c.items())}

# Esempio d'uso (da lanciare sul cluster dove esistono le feature):
# conta_per_classe('features_imagenet/hmdb51/index.csv')
print("Funzione pronta: conta_per_classe('features_imagenet/<dataset>/index.csv')")


Funzione pronta: conta_per_classe('features_imagenet/<dataset>/index.csv')


## 2. Il backbone: ResNet-50 ImageNet "inflated" a 3D (I3D)

### 2.1 Perche non un backbone pre-addestrato su Kinetics

Inizialmente avevamo usato un backbone **R3D-18 pre-addestrato su Kinetics-400**.
Poichè il nostro target è un sottoinsieme di Kinetics, questo introduceva un
**information leakage**: le feature erano già allineate al target, gonfiando
l'accuratezza zero-shot e lasciando un domain shift minimo da correggere. In
pratica la domain adaptation non aveva un vero problema da risolvere.

Per ottenere una valutazione onesta usiamo un backbone che **non ha mai visto
Kinetics**: una **ResNet-50 pre-addestrata su ImageNet** (immagini statiche).

### 2.2 Cos'e l'inflation (I3D)

ImageNet fornisce una rete **2D** (lavora su singole immagini). Per processare
clip video serve una rete **3D**. La tecnica I3D (Carreira & Zisserman, 2017)
"gonfia" ogni filtro convoluzionale 2D di dimensione k x k in un filtro 3D
t x k x k, **trasferendo i pesi** appresi su ImageNet.

Usiamo l'inizializzazione **centrata**: la fetta temporale centrale del kernel 3D
riceve i pesi 2D, le altre fette sono azzerate. Cosi la rete 3D, all'inizio,
riproduce il comportamento della rete 2D su input replicati nel tempo, e i pesi
ImageNet vengono preservati.

Il backbone inflated produce feature da **2048** dimensioni.


In [6]:
import torch
from torchvision.models import resnet50
from src.models.inflated_resnet import InflatedResNet50, inflate_conv

# Costruiamo la ResNet-50 2D (qui a pesi casuali per illustrazione; nel progetto
# si caricano i pesi ImageNet) e la "gonfiamo" a 3D.
resnet2d = resnet50(weights=None)
model3d = InflatedResNet50(resnet2d).eval()
print("Dimensione feature in output:", model3d.feature_dim)


Dimensione feature in output: 2048


In [7]:
# Verifica del trasferimento pesi: la slice temporale centrale del conv1 3D
# deve contenere esattamente i pesi 2D, il resto deve essere a zero.
w2d = resnet2d.conv1.weight.data            # (out, in, kh, kw)
w3d = model3d.conv1.weight.data             # (out, in, T=3, kh, kw)
centro = w3d.shape[2] // 2

uguali = torch.allclose(w3d[:, :, centro], w2d)
laterali_zero = torch.allclose(w3d[:, :, 0], torch.zeros_like(w3d[:, :, 0]))
print("Slice centrale = pesi 2D ImageNet:", uguali)
print("Slice laterali azzerate:", laterali_zero)


Slice centrale = pesi 2D ImageNet: True
Slice laterali azzerate: True


In [8]:
# Forward su una clip finta: (batch, canali, tempo, altezza, larghezza)
clip = torch.randn(2, 3, 16, 224, 224)   # 2 clip, 16 frame, 224x224
with torch.no_grad():
    feat = model3d(clip)
print("Input :", tuple(clip.shape))
print("Output:", tuple(feat.shape), "-> vettore di feature da 2048 dim per clip")


Input : (2, 3, 16, 224, 224)
Output: (2, 2048) -> vettore di feature da 2048 dim per clip


La funzione `build_inflated_resnet50` incapsula la costruzione e, sul cluster
offline, carica i pesi ImageNet da un file locale (scaricato da casa, dato che il
cluster non ha accesso a internet):

```python
from src.models.inflated_resnet import build_inflated_resnet50
model = build_inflated_resnet50(weights_path='weights/resnet50_imagenet.pth')
```


## 3. Pipeline di estrazione delle feature

### 3.1 Strategia

Non addestriamo end-to-end sui video: usiamo il backbone **congelato** per
estrarre, **una sola volta**, un vettore di feature per ogni clip, e salviamo i
vettori su disco. Tutto l'addestramento di domain adaptation girerà poi su questi
vettori leggeri. Vantaggi: nessuna decodifica video nel ciclo di training,
iterazioni molto rapide, e coerenza con la natura del problema (la DA allinea
distribuzioni, non impara feature visive di basso livello).

### 3.2 Lettura delle clip

I tre dataset hanno formati diversi, gestiti automaticamente:
- **HMDB-51**: ogni clip è una cartella di frame JPG -> lettura diretta delle
  immagini, campionando 16 frame uniformemente.
- **UCF-101 / Kinetics**: file video -> lettura in **streaming** con PyAV,
  prelevando solo i 16 frame necessari senza caricare l'intero video in memoria
  (evita problemi di RAM su video lunghi).


In [9]:
# Pre-processing applicato a ogni clip prima del backbone (costanti ImageNet)
from src.data.extract_features import _imagenet_video_transform

transform = _imagenet_video_transform()
# input atteso: (canali, tempo, altezza, larghezza) uint8
clip_uint8 = torch.randint(0, 256, (3, 16, 120, 160), dtype=torch.uint8)
clip_pronta = transform(clip_uint8)
print("Dopo il transform:", tuple(clip_pronta.shape), clip_pronta.dtype)
print("(ridimensionata a 224x224 e normalizzata con media/std di ImageNet)")


Dopo il transform: (3, 16, 224, 224) torch.float32
(ridimensionata a 224x224 e normalizzata con media/std di ImageNet)


### 3.3 Estrazione e salvataggio

Per ogni clip: si campionano 16 frame, si applica il transform, si esegue un
forward sul backbone congelato e si salva il vettore di feature come file `.npy`.
Viene prodotto anche un `index.csv` che mappa ogni file di feature alla sua
etichetta e al dominio.

Il comando reale eseguito sul cluster (per ciascun dataset) è:

```bash
python -m src.data.extract_features \
    --dataset hmdb51 \
    --video-root data/HMDB51 \
    --backbone inflated_resnet50 \
    --weights-path weights/resnet50_imagenet.pth \
    --out-root features_imagenet
```

Layout dell'output:
```
features_imagenet/<dataset>/<classe>/<id_clip>.npy   # vettore (2048,)
features_imagenet/<dataset>/index.csv                # path, label, domain
```


In [10]:
# Schema logico del ciclo di estrazione (versione semplificata a scopo illustrativo)
pseudocodice = '''
model, transform, feat_dim = build_backbone('inflated_resnet50', device, weights_path)

for (clip_path, classe, kind) in trova_clip(dataset):
    if kind == 'frames':
        clip = leggi_frame_da_cartella(clip_path, 16)   # HMDB
    else:
        clip = leggi_video_streaming(clip_path, 16)      # UCF, Kinetics
    x = transform(clip)                  # (C, T, 224, 224) normalizzata
    with torch.no_grad():
        feat = model(x.unsqueeze(0))     # (1, 2048)
    salva_npy(feat, classe)              # -> features_imagenet/<dataset>/<classe>/<id>.npy
'''
print(pseudocodice)



model, transform, feat_dim = build_backbone('inflated_resnet50', device, weights_path)

for (clip_path, classe, kind) in trova_clip(dataset):
    if kind == 'frames':
        clip = leggi_frame_da_cartella(clip_path, 16)   # HMDB
    else:
        clip = leggi_video_streaming(clip_path, 16)      # UCF, Kinetics
    x = transform(clip)                  # (C, T, 224, 224) normalizzata
    with torch.no_grad():
        feat = model(x.unsqueeze(0))     # (1, 2048)
    salva_npy(feat, classe)              # -> features_imagenet/<dataset>/<classe>/<id>.npy



### 3.4 Risultato

L'estrazione produce, per i tre dataset, le feature da 2048 dimensioni usate in
tutta la seconda parte del progetto:

| Dataset | Clip con feature | Dim. feature |
| :--- | :---: | :---: |
| HMDB-51 | 1684 | 2048 |
| UCF-101 | 1457 | 2048 |
| Kinetics | 303 | 2048 |

A questo punto la **Parte 1 è completa**: abbiamo lo spazio di etichette condiviso,
il backbone privo di leakage, e le feature estratte e salvate. La Parte 2 userà
queste feature per addestrare il modello multi-source (encoder condiviso,
classificatori per sorgente, allineamento adversariale con GRL, ensemble pesato) e
per la valutazione sul target.
